# GRUEncoder Training — SOLUSDT Phase A

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sandeep999-cyber/solusdt-ml-trader/blob/main/colab_train.ipynb)

### Workflow
1. **Open** via the badge above (always gets latest notebook from GitHub)
2. **Runtime → Change runtime type → T4 GPU**
3. **Run All** (Cell 0 → Cell 7)
4. Locally: leave `python scripts/watch_checkpoints.py` running — it auto-syncs + analyzes

### Drive layout (one-time)
```
MyDrive/ModelProject/
  year=2023/ ... year=2024/   # feature parquet (already there)
  .version
  checkpoints/                # created by Cell 7
```

In [ ]:
# Cell 0: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# Cell 1: Download code + link data from Drive
import json, os, urllib.request, tarfile, shutil, time

REPO = 'sandeep999-cyber/solusdt-ml-trader'
PROJECT_DIR = '/content/ModelProject'
ARCHIVE_URL = f'https://github.com/{REPO}/archive/main.tar.gz'
API_COMMIT_URL = f'https://api.github.com/repos/{REPO}/commits/main'

# Drive paths — data lives at ModelProject root (year=YYYY partitions)
DRIVE_ROOT = '/content/drive/MyDrive/ModelProject'
DRIVE_DATA = DRIVE_ROOT  # year=2023/, year=2024/, .version are here
DRIVE_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints')

# --- Download latest code from GitHub tarball ---
if os.path.exists(PROJECT_DIR):
    runs_dir = os.path.join(PROJECT_DIR, 'model', 'runs')
    if os.path.isdir(runs_dir):
        entries = [e for e in os.listdir(runs_dir) if e not in ('__pycache__',)]
        if entries:
            print(f'WARNING: wiping {len(entries)} local run(s) in model/runs/.')
            print('Only continue if Cell 7 already saved them to Drive.')
    shutil.rmtree(PROJECT_DIR)

archive_path = '/tmp/repo.tar.gz'
if os.path.exists(archive_path):
    os.remove(archive_path)
print(f'Downloading {ARCHIVE_URL}...')
urllib.request.urlretrieve(ARCHIVE_URL, archive_path)
print('Extracting...')
with tarfile.open(archive_path) as tar:
    tar.extractall('/content')

extracted = f'/content/{REPO.split("/")[1]}-main'
if not os.path.isdir(extracted):
    raise RuntimeError(f'Expected extract dir missing: {extracted}')
os.rename(extracted, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Code ready at {PROJECT_DIR}')
print(f'  configs: {os.listdir("configs")}')

# --- Record git commit ---
# Fallback: fetch from GitHub API (no .git in tarball)
git_commit = 'unknown'
try:
    req = urllib.request.urlopen(API_COMMIT_URL, timeout=10)
    data = json.loads(req.read().decode())
    git_commit = data['sha'][:10]  # short SHA
    print(f'Git commit: {git_commit}')
except Exception as e:
    print(f'Could not fetch commit SHA: {e}')
os.environ['GIT_COMMIT'] = git_commit

# --- Enable mid-run Drive mirror ---
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT
print(f'Drive checkpoint mirror: {DRIVE_CKPT}')

# --- Link processed data ---
# Expected layout on Drive: ModelProject/year=2023/, year=2024/, .version
LOCAL_DATA = os.path.join(PROJECT_DIR, 'data', 'processed', 'v1', 'SOLUSDT', '1m')

def _has_feature_data(path):
    if not os.path.isdir(path):
        return False
    return any(name.startswith('year=') for name in os.listdir(path))

def _drive_fingerprint():
    parts = sorted(os.listdir(DRIVE_DATA))
    info = []
    for p in parts:
        full = os.path.join(DRIVE_DATA, p)
        if os.path.isdir(full):
            mtime = os.path.getmtime(full)
            size = sum(
                os.path.getsize(os.path.join(dirpath, f))
                for dirpath, _, filenames in os.walk(full)
                for f in filenames if f.endswith('.parquet')
            )
            info.append((p, mtime, size))
    return json.dumps(info, sort_keys=True)

do_copy = True
if _has_feature_data(LOCAL_DATA):
    fp_path = os.path.join(LOCAL_DATA, '.drive_fingerprint')
    if os.path.exists(fp_path):
        prev = open(fp_path).read()
        curr = _drive_fingerprint()
        if prev == curr:
            do_copy = False
            print('Data fingerprint unchanged — skipping Drive copy')

if do_copy:
    if not _has_feature_data(DRIVE_DATA):
        raise FileNotFoundError(
            f'Feature data not found at {DRIVE_DATA}.\n'
            'Expected year=YYYY/month=MM/features.parquet under MyDrive/ModelProject/.\n'
            'Current Drive contents: '
            + str(os.listdir(DRIVE_ROOT) if os.path.isdir(DRIVE_ROOT) else 'MISSING')
        )
    os.makedirs(os.path.dirname(LOCAL_DATA), exist_ok=True)
    if os.path.lexists(LOCAL_DATA):
        if os.path.islink(LOCAL_DATA) or os.path.isfile(LOCAL_DATA):
            os.unlink(LOCAL_DATA)
        else:
            shutil.rmtree(LOCAL_DATA)
    # Copy to local SSD (not symlink). Drive latency only hits Parquet load;
    # after CausalWindowDataset builds tensors, training is pure RAM.
    # Only copy year=YYYY partitions, not junk files in Drive root.
    print('Copying feature data Drive -> local SSD (one-time, ~1-2 min)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for item in os.listdir(DRIVE_DATA):
        if item.startswith('year='):
            src = os.path.join(DRIVE_DATA, item)
            dst = os.path.join(LOCAL_DATA, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Copied data to {LOCAL_DATA}')
    # Save fingerprint for next session
    with open(os.path.join(LOCAL_DATA, '.drive_fingerprint'), 'w') as f:
        f.write(_drive_fingerprint())
else:
    print(f'Data available at {LOCAL_DATA}')

# Remove any junk files (colab_train.ipynb, etc.) copied from Drive in prior runs
for item in os.listdir(LOCAL_DATA):
    if not item.startswith('year=') and item != '.version':
        path = os.path.join(LOCAL_DATA, item)
        if os.path.isfile(path) or os.path.islink(path):
            os.remove(path)
        else:
            shutil.rmtree(path)

assert _has_feature_data(LOCAL_DATA), 'Data setup failed — no year= partitions'
print(f'Data partitions: {[n for n in os.listdir(LOCAL_DATA) if n.startswith("year=")]}')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')

In [ ]:
# Cell 3: Install deps (torch is preinstalled on Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'pyarrow', 'pyyaml'])
print('Ready.')

In [ ]:
# Cell 4: Smoke test (~10s)
import subprocess, sys

def run_train(config, extra_args=None):
    cmd = [sys.executable, '-m', 'model.train', '--config', config]
    if extra_args:
        cmd.extend(extra_args)
    print('>', ' '.join(cmd))
    result = subprocess.run(cmd, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        if result.stderr:
            sys.stderr.write(result.stderr)
        raise RuntimeError(f'Training failed (exit {result.returncode}). See output above.')
    return True

smoke_ok = False
try:
    run_train('configs/phase_a_gru_h32.yaml', ['--smoke-test-only'])
    smoke_ok = True
    print('\nSmoke test PASSED.')
except Exception as e:
    print(f'Smoke test FAILED: {e}')


In [ ]:
# Cell 5: Full training — GRU h32, 30 epochs (~5 min on T4)
if smoke_ok:
    run_train('configs/phase_a_gru_h32.yaml')
else:
    print('Skipped — smoke test failed.')

---
### Diagnostic: fixed-variance GRU

Forces `log_var=0` so loss = plain MSE. If MSE drops below ~1.016,
the flat mean is a variance-shortcut pathology, not a genuine ceiling.

Run 6a → 6b after Cell 5.

In [ ]:
# Cell 6a: Diagnostic smoke test
diag_smoke_ok = False
try:
    run_train('configs/diag_fixed_var.yaml', ['--smoke-test-only'])
    diag_smoke_ok = True
    print('\nDiagnostic smoke test PASSED.')
except Exception as e:
    print(f'Diagnostic smoke test FAILED: {e}')

In [ ]:
# Cell 6b: Diagnostic full run — 15 epochs (~3 min)
if diag_smoke_ok:
    run_train('configs/diag_fixed_var.yaml')
else:
    print('Skipped — diagnostic smoke test failed.')

In [ ]:
# Cell 7: Save all runs to Drive (watcher picks these up automatically)
import json, shutil, os
from datetime import datetime, timezone
from pathlib import Path

RUNS_DIR = Path('model/runs')
if not RUNS_DIR.exists():
    raise RuntimeError('model/runs/ not found — no training was run.')

all_dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir() and d.name != '__pycache__'])
real_runs = [d for d in all_dirs if 'smoketest' not in d.name]
if not real_runs:
    real_runs = all_dirs
if not real_runs:
    raise RuntimeError('No run directories found.')

DRIVE_ROOT = '/content/drive/MyDrive/ModelProject'
DRIVE_RUN = Path(DRIVE_ROOT) / 'checkpoints'
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

saved = []
for run_dir in real_runs:
    name = run_dir.name
    best = run_dir / 'checkpoints' / 'best.pt'
    if not best.exists():
        print(f'  {name}: no best.pt — skip')
        continue

    shutil.copy2(best, DRIVE_RUN / f'{name}_best.pt')
    print(f'  {name}/best.pt -> Drive')

    config = run_dir / 'config.yaml'
    if config.exists():
        shutil.copy2(config, DRIVE_RUN / f'{name}_config.yaml')

    metrics = run_dir / 'metrics.jsonl'
    if metrics.exists():
        shutil.copy2(metrics, DRIVE_RUN / f'{name}_metrics.jsonl')
        with open(metrics) as f:
            lines = [l for l in f if l.strip()]
        if lines:
            last = json.loads(lines[-1])
            print(f'    epoch={last.get("epoch","?")}  mse={last.get("mse","?")}  loss={last.get("loss","?")}')

    summary = run_dir / 'summary.md'
    if summary.exists():
        shutil.copy2(summary, DRIVE_RUN / f'{name}_summary.md')

    saved.append(name)

if not saved:
    raise RuntimeError('No checkpoints saved.')

# Pointer for local pull/watch scripts
latest = saved[-1]
shutil.copy2(RUNS_DIR / latest / 'checkpoints' / 'best.pt', DRIVE_RUN / 'best.pt')
pointer = {
    'run_name': latest,
    'checkpoint_path': str(DRIVE_RUN / f'{latest}_best.pt'),
    'updated': datetime.now(timezone.utc).isoformat(),
    'runs_saved': saved,
}
with open(DRIVE_RUN / 'latest.json', 'w') as f:
    json.dump(pointer, f, indent=2)

print(f'\nSaved {len(saved)} run(s). Pointer -> {latest}')
print(f'Drive path: {DRIVE_RUN}')
print('Local watcher should pick these up within ~30s.')

In [ ]:
# Cell 8: Optional manual browser download
from pathlib import Path
from google.colab import files

RUNS_DIR = Path('model/runs')
dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir() and 'smoketest' not in d.name])
if not dirs:
    dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir()])
latest = dirs[-1] if dirs else None

if latest:
    best_pt = latest / 'checkpoints' / 'best.pt'
    if best_pt.exists():
        files.download(str(best_pt))
        print(f'Downloading {latest.name}/best.pt...')
    else:
        print(f'No best.pt in {latest.name}')
else:
    print('No runs found')